# PLOS ONE revision: effect-size reproduction
This notebook regenerates the new random-effects models and figures from the supplied coded patient-level inputs. It does **not** reread the large source count files; see README for that workflow. New output is isolated in `reproduced/`.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json
import pandas as pd
from IPython.display import display, Image
base = Path.cwd()
assert (base / "analysis/meta_patient_inputs.csv").is_file(), "Run from the archive root"
out = base / "reproduced"
for folder in ["analysis", "validation", "figures"]:
    (out / folder).mkdir(parents=True, exist_ok=True)
for source in (base / "analysis").glob("*.csv"):
    shutil.copy2(source, out / "analysis" / source.name)
os.environ["PLOS_REVISION_DIR"] = str(out)
sys.path.insert(0, str(base / "code"))
rs = os.environ.get("RSCRIPT") or shutil.which("Rscript")
assert rs, "Rscript is required; set RSCRIPT if necessary"
print("Input patient-level rows:", len(pd.read_csv(out / "analysis/meta_patient_inputs.csv")))

## Fit all effect-size models
REML, modified Knapp-Hartung intervals, heterogeneity and cohort omissions; patient pairing is represented in each contrast.

In [ ]:
fit = subprocess.run([rs, str(base / "code/stage64_revision_random_effects.R"), str(out)], capture_output=True, text=True, check=True)
(out / "validation/R_fit.log").write_text(fit.stdout + fit.stderr)
summary = pd.read_csv(out / "analysis/meta_summary.csv")
display(summary.loc[summary.analysis.isin(["primary_labels", "uniform_labels"]), ["analysis", "k", "n", "estimate", "ci_low", "ci_high", "p_value", "tau2", "I2", "prediction_low", "prediction_high"]])

## Independent numerical and supplied-input checks
The source-count agreement checks below read the archived audit summaries; they do not freshly inspect the external raw files.

In [ ]:
from stage65_revision_outputs import validate, setup_style, forest, preprocessing_plot, discordance_plot
checks = validate()
assert checks["failed"] == 0
reference = pd.read_csv(base / "analysis/meta_summary.csv").sort_values("analysis").reset_index(drop=True)
reproduced = summary.sort_values("analysis").reset_index(drop=True)
import numpy as np
assert np.allclose(reference[["estimate", "ci_low", "ci_high", "p_value", "tau2"]], reproduced[["estimate", "ci_low", "ci_high", "p_value", "tau2"]], atol=1e-8, rtol=1e-6)
print("All 16 reproduced models agree with the supplied reference estimates.")

## Figures
Effect means and medians are different estimands. Normalization panels have different units; eligibility changes are shown.

In [ ]:
setup_style()
forest()
preprocessing_plot()
discordance_plot()
for name in ["S15_Fig_random_effects", "S16_Fig_preprocessing", "S17_Fig_GSE162631"]:
    display(Image(filename=str(out / "figures" / (name + ".png"))))

## Interpretation limits
The original-label pooled estimate is positive. Uniform labels show heterogeneity and a prediction interval crossing zero; four of five omission intervals cross zero. GSE162631 source-count checks retain one positive and two negative patients. This notebook cannot validate source data it has not opened, establish an AC-like-to-myeloid signaling mechanism or confirm journal upload status.